# Experimento INICIAL (sin faltantes simulados)

## Carga, división y preparación

In [ ]:
!pip install hyperimpute

In [ ]:
# ================================================
# CELDA 0: LIBRERÍAS
# ================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from hyperimpute.plugins.imputers import Imputers
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# ================================================
# PARTE 1: CARGA Y PREPARACIÓN DE DATOS
# ================================================

nombre_dataset = "df"
df = pd.read_parquet(f"{nombre_dataset}.parquet")
print(f"Dataset cargado: {df.shape}")

# Variable objetivo y separación de predictores
y = LabelEncoder().fit_transform(df["Diabetes"])
X = df.drop(columns=["Diabetes"])

# División estratificada
X_train, X_test, y_train_arr, y_test_arr = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
y_train = pd.Series(y_train_arr, index=X_train.index)
y_test = pd.Series(y_test_arr, index=X_test.index)

# Identificación de variables categóricas y numéricas
categorical_vars = X.select_dtypes(include="category").columns.tolist()
numerical_vars = X.select_dtypes(include=["float64", "int32"]).columns.tolist()

print(f"Variables categóricas ({len(categorical_vars)}): {categorical_vars}")
print(f"Variables numéricas ({len(numerical_vars)}): {numerical_vars}")

## Funciones de imputación

In [ ]:
# ================================================
# PARTE 2: IMPUTACIÓN
# ================================================

def apply_imputation_method(X_train, X_test, method='simple_mean', categorical_vars=[], numerical_vars=[]):
    X_train_imp = X_train.copy()
    X_test_imp = X_test.copy()

    if method in ['simple_mean', 'simple_median', 'knn']:
        # Imputación para categóricas: siempre con moda
        if categorical_vars:
            imputer_cat = SimpleImputer(strategy='most_frequent')
            X_train_imp[categorical_vars] = imputer_cat.fit_transform(X_train_imp[categorical_vars])
            X_test_imp[categorical_vars] = imputer_cat.transform(X_test_imp[categorical_vars])

        # Imputación para numéricas
        if method == 'simple_mean':
            imputer_num = SimpleImputer(strategy='mean')
        elif method == 'simple_median':
            imputer_num = SimpleImputer(strategy='median')
        elif method == 'knn':
            imputer_num = KNNImputer(n_neighbors=5)

        if numerical_vars:
            X_train_imp[numerical_vars] = imputer_num.fit_transform(X_train_imp[numerical_vars])
            X_test_imp[numerical_vars] = imputer_num.transform(X_test_imp[numerical_vars])

    elif method in ['gain', 'missforest', 'mice']:
        # Unir y asegurar tipos numéricos (strings a códigos)
        X_all = pd.concat([X_train, X_test]).sort_index()
        X_all_clean = X_all.copy()

        for col in X_all_clean.columns:
            if X_all_clean[col].dtype == 'object' or str(X_all_clean[col].dtype).startswith('category'):
                X_all_clean[col] = pd.Series(X_all_clean[col], dtype="category").cat.codes

        imputador = Imputers().get(method)
        X_all_imp = imputador.fit_transform(X_all_clean)

        X_train_imp = X_all_imp.loc[X_train.index]
        X_test_imp = X_all_imp.loc[X_test.index]

    else:
        raise ValueError(f"Método de imputación no reconocido: {method}")

    return X_train_imp, X_test_imp


## Funciones de codificación, balanceo y evaluación

In [ ]:
# ================================================
# PARTE 3: CODIFICACIÓN Y EVALUACIÓN
# ================================================

def encode_categorical_variables(X_train, X_test, categorical_vars):
    X_train_encoded = X_train.copy()
    X_test_encoded = X_test.copy()
    encoders = {}

    for var in categorical_vars:
        le = LabelEncoder()
        X_train_encoded[var] = le.fit_transform(X_train_encoded[var].astype(str))
        encoders[var] = le

        test_vals = X_test_encoded[var].astype(str)
        test_encoded = []
        for val in test_vals:
            if val in le.classes_:
                test_encoded.append(le.transform([val])[0])
            else:
                most_frequent = X_train[var].mode()[0] if not X_train[var].empty else le.classes_[0]
                test_encoded.append(le.transform([str(most_frequent)])[0])

        X_test_encoded[var] = test_encoded

    return X_train_encoded, X_test_encoded, encoders

def train_and_evaluate(X_train, X_test, y_train, y_test, method_name, balance_method='none'):
    print(f"\n--- MÉTODO: {method_name.upper()} | BALANCE: {balance_method.upper()} ---")

    X_train_bal = X_train.copy()
    y_train_bal = y_train.copy()

    if balance_method == 'smote':
        smote = SMOTE(random_state=42)
        X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
        print(f"Después de SMOTE: {np.bincount(y_train_bal)}")

    elif balance_method == 'undersample':
        rus = RandomUnderSampler(random_state=42)
        X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)
        print(f"Después de Undersampling: {np.bincount(y_train_bal)}")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_bal)
    X_test_scaled = scaler.transform(X_test)

    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'KNN': KNeighborsClassifier(n_neighbors=5)
    }

    results = {}

    for model_name, model in models.items():
        if balance_method == 'none' and hasattr(model, 'class_weight'):
            model.set_params(class_weight='balanced')

        model.fit(X_train_scaled, y_train_bal)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]

        auc = roc_auc_score(y_test, y_proba)
        f1 = f1_score(y_test, y_pred)
        pr_auc = average_precision_score(y_test, y_proba)
        cv_auc = cross_val_score(model, X_train_scaled, y_train_bal,
                                 cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                 scoring='roc_auc')

        results[model_name] = {
            'AUC_test': auc,
            'AUC_cv_mean': cv_auc.mean(),
            'AUC_cv_std': cv_auc.std(),
            'F1': f1,
            'PR_AUC': pr_auc
        }

        print(f"{model_name}:")
        print(f"  AUC Test: {auc:.4f}")
        print(f"  AUC CV: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
        print(f"  F1: {f1:.4f}")
        print(f"  PR AUC: {pr_auc:.4f}")

    return results


## Bucle maestro de ejecución (42 combinaciones)

In [ ]:
# ================================================
# PARTE 4: BUCLE PRINCIPAL DE EXPERIMENTOS
# ================================================

metodos_imputacion = [
    'simple_mean', 'simple_median', 'knn',
    'gain', 'missforest', 'mice'
]

estrategias_balanceo = ['none', 'smote']

all_results = {}

for metodo in tqdm(metodos_imputacion, desc="Imputación", disable=True):
    print(f"\n🔄 APLICANDO IMPUTACIÓN: {metodo.upper()}")

    X_train_imp, X_test_imp = apply_imputation_method(
        X_train, X_test, metodo, categorical_vars, numerical_vars
    )
    y_train_imp = y_train
    y_test_imp = y_test

    if metodo in ['simple_mean', 'simple_median', 'knn']:
        X_train_imp, X_test_imp, _ = encode_categorical_variables(
            X_train_imp, X_test_imp, categorical_vars
        )

    for balance in estrategias_balanceo:
        key = f"{metodo}_{balance}"
        resultados = train_and_evaluate(
            X_train_imp, X_test_imp, y_train_imp, y_test_imp,
            method_name=metodo, balance_method=balance
        )
        all_results[key] = resultados


## Visualización y guardado

In [ ]:
# ================================================
# PARTE 5: VISUALIZACIÓN Y GUARDADO DE RESULTADOS
# ================================================

results_summary = []

for exp_key, models_results in all_results.items():
    metodo, balance = exp_key.rsplit('_', 1)
    for modelo, metrics in models_results.items():
        results_summary.append({
            'Imputación': metodo,
            'Balance': balance,
            'Modelo': modelo,
            'AUC_Test': metrics['AUC_test'],
            'AUC_CV_Mean': metrics['AUC_cv_mean'],
            'AUC_CV_Std': metrics['AUC_cv_std'],
            'F1': metrics['F1'],
            'PR_AUC': metrics['PR_AUC']
        })

results_df = pd.DataFrame(results_summary)
results_df = results_df.sort_values(by='F1', ascending=False)

# Guardar como parquet
results_df.to_parquet(f"{nombre_dataset}_resultados.parquet", index=False)
print(f"\n✅ Resultados guardados en {nombre_dataset}_resultados.parquet")

# Visualización
results_df["Etiqueta"] = results_df.apply(
    lambda row: f"{row['Imputación']}_{row['Balance']} - {row['Modelo']}", axis=1
)

plt.figure(figsize=(14, 8))
sns.barplot(data=results_df, y="Etiqueta", x="F1", hue="Modelo", dodge=False)
plt.title("Comparativa F1 por imputación, modelo y balance")
plt.tight_layout()
plt.legend().set_visible(False)
plt.show()

# Heatmaps
for metric, cmap, title in [
    ("F1", "YlGnBu", "F1-score"),
    ("AUC_Test", "Blues", "AUC Test"),
    ("PR_AUC", "Oranges", "PR AUC")
]:
    pivot = results_df.pivot_table(index="Imputación", columns="Modelo", values=metric, aggfunc="max")
    plt.figure(figsize=(8, 5))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap=cmap, cbar_kws={"label": metric})
    plt.title(f"{title} máximo por imputación y modelo")
    plt.tight_layout()
    plt.show()



# === Guardar resultados ===
output_file = f"{nombre_dataset}_resultados.parquet"
results_df.to_parquet(output_file, index=False)
print(f"\n✅ Resultados guardados como: {output_file}")

# === Imprimir top combinaciones ===
print("\n🔝 TOP 5 combinaciones por F1-score:")
display(results_df.head(5).round(4))
